# AdventureWorks — Transformación Bronze → Silver
## Microsoft Fabric | Data Engineering
Dataset: AdventureWorks Sales (Microsoft)
Objetivo: limpieza, tipado y normalización de tablas para modelo en estrella

In [7]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

fact_sales     = spark.read.option("header","true").option("inferSchema","true").csv("Files/bronze/FactSales.csv")
dim_date       = spark.read.option("header","true").option("inferSchema","true").csv("Files/bronze/DimDate.csv")
dim_product    = spark.read.option("header","true").option("inferSchema","true").csv("Files/bronze/DimProduct.csv")
dim_customer   = spark.read.option("header","true").option("inferSchema","true").csv("Files/bronze/DimCustomer.csv")
dim_territory  = spark.read.option("header","true").option("inferSchema","true").csv("Files/bronze/DimTerritory.csv")
dim_reseller   = spark.read.option("header","true").option("inferSchema","true").csv("Files/bronze/DimReseller.csv")
dim_salesorder = spark.read.option("header","true").option("inferSchema","true").csv("Files/bronze/DimSalesOrder.csv")

print(" -> CSVs cargados correctamente desde Bronze")
fact_sales.printSchema()

StatementMeta(, 046f1fe0-7594-4b64-b060-56ffb9b11474, 10, Finished, Available, Finished, False)

 -> CSVs cargados correctamente desde Bronze
root
 |-- SalesOrderLineKey: integer (nullable = true)
 |-- ResellerKey: integer (nullable = true)
 |-- CustomerKey: integer (nullable = true)
 |-- ProductKey: integer (nullable = true)
 |-- OrderDateKey: integer (nullable = true)
 |-- DueDateKey: integer (nullable = true)
 |-- ShipDateKey: integer (nullable = true)
 |-- SalesTerritoryKey: integer (nullable = true)
 |-- Order Quantity: integer (nullable = true)
 |-- Unit Price: double (nullable = true)
 |-- Extended Amount: double (nullable = true)
 |-- Unit Price Discount Pct: integer (nullable = true)
 |-- Product Standard Cost: double (nullable = true)
 |-- Total Product Cost: double (nullable = true)
 |-- Sales Amount: double (nullable = true)



## 1. Limpieza de FactSales
Transformaciones aplicadas:
- Eliminación de registros inválidos (CustomerKey = -1 y ResellerKey = -1)
- Conversión de claves de fecha (yyyyMMdd) a tipo Date real
- Eliminación de duplicados por SalesOrderLineKey
- Renombrado de columnas a nombres limpios sin espacios

In [8]:
fact_silver = fact_sales \
    .filter(~((F.col("CustomerKey") == -1) & (F.col("ResellerKey") == -1))) \
    .dropDuplicates(["SalesOrderLineKey"]) \
    .withColumn("OrderDate", F.to_date(F.col("OrderDateKey").cast("string"), "yyyyMMdd")) \
    .withColumn("DueDate",   F.to_date(F.col("DueDateKey").cast("string"),   "yyyyMMdd")) \
    .withColumn("ShipDate",  F.to_date(F.col("ShipDateKey").cast("string"),  "yyyyMMdd")) \
    .withColumnRenamed("Order Quantity",          "OrderQuantity") \
    .withColumnRenamed("Unit Price",              "UnitPrice") \
    .withColumnRenamed("Extended Amount",         "ExtendedAmount") \
    .withColumnRenamed("Unit Price Discount Pct", "DiscountPct") \
    .withColumnRenamed("Sales Amount",            "SalesAmount") \
    .withColumnRenamed("Total Product Cost",      "TotalCost") \
    .select("SalesOrderLineKey","CustomerKey","ResellerKey","ProductKey",
            "SalesTerritoryKey","OrderDate","DueDate","ShipDate",
            "OrderQuantity","UnitPrice","DiscountPct","SalesAmount","TotalCost")

print(f" -> FactSales Silver: {fact_silver.count()} filas")
fact_silver.show(5)

StatementMeta(, 046f1fe0-7594-4b64-b060-56ffb9b11474, 11, Finished, Available, Finished, False)

 -> FactSales Silver: 121253 filas
+-----------------+-----------+-----------+----------+-----------------+----------+----------+----------+-------------+---------+-----------+-----------+---------+
|SalesOrderLineKey|CustomerKey|ResellerKey|ProductKey|SalesTerritoryKey| OrderDate|   DueDate|  ShipDate|OrderQuantity|UnitPrice|DiscountPct|SalesAmount|TotalCost|
+-----------------+-----------+-----------+----------+-----------------+----------+----------+----------+-------------+---------+-----------+-----------+---------+
|         43675001|         -1|        670|       324|                3|2017-07-17|2017-07-27|2017-07-24|            4| 419.4589|          0|  1677.8356|1652.5852|
|         43680011|         -1|        491|       314|                4|2017-07-18|2017-07-28|2017-07-25|            2| 2146.962|          0|   4293.924|4342.5884|
|         43682004|         -1|        486|       342|                2|2017-07-19|2017-07-29|2017-07-26|            2| 419.4589|          0|   8

## 2. Limpieza de dimensiones
Transformaciones aplicadas por tabla:
- **DimProduct**: tipado de precios a Double, eliminación de duplicados
- **DimCustomer**: filtrado de registros [Not Applicable], renombrado de columnas
- **DimTerritory**: eliminación de duplicados
- **DimReseller**: filtrado de registros [Not Applicable], renombrado de columnas
- **DimDate**: conversión de columna Date a tipo DateType real

In [9]:
dim_product_silver = dim_product \
    .filter(F.col("ProductKey").isNotNull()) \
    .withColumn("StandardCost", F.col("Standard Cost").cast(DoubleType())) \
    .withColumn("ListPrice",    F.col("List Price").cast(DoubleType())) \
    .drop("Standard Cost", "List Price") \
    .dropDuplicates(["ProductKey"])

dim_customer_silver = dim_customer \
    .filter(F.col("Customer") != "[Not Applicable]") \
    .dropDuplicates(["CustomerKey"]) \
    .withColumnRenamed("Customer ID",    "CustomerID") \
    .withColumnRenamed("State-Province", "StateProvince") \
    .withColumnRenamed("Country-Region", "Country") \
    .withColumnRenamed("Postal Code",    "PostalCode")

dim_territory_silver = dim_territory \
    .dropDuplicates(["SalesTerritoryKey"])

dim_reseller_silver = dim_reseller \
    .filter(F.col("Reseller") != "[Not Applicable]") \
    .withColumnRenamed("Reseller ID",    "ResellerID") \
    .withColumnRenamed("Business Type",  "BusinessType") \
    .withColumnRenamed("State-Province", "StateProvince") \
    .withColumnRenamed("Country-Region", "Country") \
    .withColumnRenamed("Postal Code",    "PostalCode") \
    .dropDuplicates(["ResellerKey"])

dim_date_silver = dim_date \
    .withColumn("Date", F.col("Date").cast(DateType())) \
    .dropDuplicates(["DateKey"])

print(" -> Dimensiones limpias:")
print(f"   DimProduct:   {dim_product_silver.count()} filas")
print(f"   DimCustomer:  {dim_customer_silver.count()} filas")
print(f"   DimDate:      {dim_date_silver.count()} filas")
print(f"   DimTerritory: {dim_territory_silver.count()} filas")
print(f"   DimReseller:  {dim_reseller_silver.count()} filas")

StatementMeta(, 046f1fe0-7594-4b64-b060-56ffb9b11474, 12, Finished, Available, Finished, False)

 -> Dimensiones limpias:
   DimProduct:   397 filas
   DimCustomer:  18484 filas
   DimDate:      1461 filas
   DimTerritory: 11 filas
   DimReseller:  701 filas


## 3. Escritura de tablas Silver en el Lakehouse
Las tablas se guardan en formato Delta Lake en la capa `Tables/` del Lakehouse.
Esto las hace accesibles directamente desde Power BI Desktop mediante el conector de Fabric.

In [10]:
from pyspark.sql import functions as F
import re

def clean_col_names(df):
    """Limpia nombres de columnas: elimina caracteres especiales"""
    for col in df.columns:
        clean = re.sub(r'[,;{}()\n\t= ]', '_', col).strip('_')
        if clean != col:
            df = df.withColumnRenamed(col, clean)
    return df

fact_silver_clean      = clean_col_names(fact_silver)
dim_product_clean      = clean_col_names(dim_product_silver)
dim_customer_clean     = clean_col_names(dim_customer_silver)
dim_territory_clean    = clean_col_names(dim_territory_silver)
dim_reseller_clean     = clean_col_names(dim_reseller_silver)
dim_date_clean         = clean_col_names(dim_date_silver)

fact_silver_clean.write.mode("overwrite").format("delta").saveAsTable("FactSales")
dim_product_clean.write.mode("overwrite").format("delta").saveAsTable("DimProduct")
dim_customer_clean.write.mode("overwrite").format("delta").saveAsTable("DimCustomer")
dim_territory_clean.write.mode("overwrite").format("delta").saveAsTable("DimTerritory")
dim_reseller_clean.write.mode("overwrite").format("delta").saveAsTable("DimReseller")
dim_date_clean.write.mode("overwrite").format("delta").saveAsTable("DimDate")

print(" - Tablas Delta guardadas en el Lakehouse")
print("   Listas para conectar desde Power BI Desktop")

StatementMeta(, 046f1fe0-7594-4b64-b060-56ffb9b11474, 13, Finished, Available, Finished, False)

 - Tablas Delta guardadas en el Lakehouse
   Listas para conectar desde Power BI Desktop


In [11]:
# Ver qué tablas existen en el Lakehouse
tables = spark.sql("SHOW TABLES").collect()
print(f"Tablas encontradas: {len(tables)}")
for t in tables:
    print(f"  {t}")

StatementMeta(, 046f1fe0-7594-4b64-b060-56ffb9b11474, 15, Finished, Available, Finished, False)

Tablas encontradas: 6
  Row(namespace='IvanDelgado.LH_AdventureWorks.dbo', tableName='dimcustomer', isTemporary=False)
  Row(namespace='IvanDelgado.LH_AdventureWorks.dbo', tableName='dimdate', isTemporary=False)
  Row(namespace='IvanDelgado.LH_AdventureWorks.dbo', tableName='dimproduct', isTemporary=False)
  Row(namespace='IvanDelgado.LH_AdventureWorks.dbo', tableName='dimreseller', isTemporary=False)
  Row(namespace='IvanDelgado.LH_AdventureWorks.dbo', tableName='dimterritory', isTemporary=False)
  Row(namespace='IvanDelgado.LH_AdventureWorks.dbo', tableName='factsales', isTemporary=False)


##  Pipeline Bronze → Silver completado
| Tabla | Capa | Formato |
|---|---|---|
| FactSales | Silver | Delta Lake |
| DimProduct | Silver | Delta Lake |
| DimCustomer | Silver | Delta Lake |
| DimTerritory | Silver | Delta Lake |
| DimReseller | Silver | Delta Lake |
| DimDate | Silver | Delta Lake |